In [11]:
import pandas as pd
import spacy
from tqdm import tqdm
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# === Input ===
INPUT_CSV = "/Users/zphilipp/Downloads/search_queries.csv"
QUERY_COLUMN = "query"
DATE_COLUMN = "eventdate"

# === Output ===
ENTITIES_OUTPUT_CSV = "q2_location_entities.csv"
QUERIES_WITH_ENTITIES_CSV = "q2_queries_with_locations.csv"

# Step 1: load data for Q2 ===
print("Reading data...")
df = pd.read_csv(INPUT_CSV, parse_dates=[DATE_COLUMN])
df_q2 = df[df[DATE_COLUMN].dt.month.isin([4, 5, 6])].copy()
df_q2[QUERY_COLUMN] = df_q2[QUERY_COLUMN].fillna("").str.lower()
print(f"Počet dotazů v Q2: {len(df_q2):,}")
df_q2.head()

Reading data...
Počet dotazů v Q2: 12,704,541


,eventdate,division,query,search_count,clicks,orders
72338,2024-05-21,Minneapolis / St Paul,liposuction-surgery,1,0,0
72435,2024-04-01,abbotsford,massage,2,1,0
72436,2024-04-02,abbotsford,oil change,1,0,0
72437,2024-04-04,abbotsford,bartending,1,1,0
72438,2024-04-04,abbotsford,las vegas,1,0,0


In [12]:
nlp = spacy.load("en_core_web_sm")

# === Query analyze ===
print("I’m analyzing queries using ... NER...")
queries = df_q2[QUERY_COLUMN].dropna().unique().tolist()
location_entities = []
queries_with_entity = []

for doc in tqdm(nlp.pipe(queries, batch_size=100), total=len(queries)):
    ents = [ent.text.lower() for ent in doc.ents if ent.label_ in ("GPE", "LOC")]
    if ents:
        location_entities.extend(ents)
        queries_with_entity.append(doc.text)

# === Step 4: Outputs ===
total_queries = len(queries)
matched_queries = len(queries_with_entity)
coverage_percent = round(100 * matched_queries / total_queries, 2)

print(f"\n✅ Coverage: {matched_queries:,} / {total_queries:,} queries ({coverage_percent} % containts location)")

# Outpup 1: list of entities
entity_df = pd.Series(location_entities).value_counts().reset_index()
entity_df.columns = ["Location Entity", "Count"]
entity_df.to_csv(ENTITIES_OUTPUT_CSV, index=False)
print(f"📄 Entity saved to: {ENTITIES_OUTPUT_CSV}")
query_df.head(100)

I’m analyzing queries using ... NER...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2074970/2074970 [22:28<00:00, 1539.02it/s]


✅ Coverage: 212,385 / 2,074,970 queries (10.24 % containts location)
📄 Entity saved to: q2_location_entities.csv


,query
0,las vegas
1,mexico hotels all inclusive
2,typhoon texas waterpark
3,spa houston
4,dreamhack dallas
5,kalahari
6,dallas cowboys
7,"great wolf lodge dallas, tx"
8,greece
9,resorts texas


In [13]:
# Output 2: list of queries
query_df = pd.DataFrame({QUERY_COLUMN: queries_with_entity})
query_df.to_csv(QUERIES_WITH_ENTITIES_CSV, index=False)
print(f"📄 Queries with locality saved as: {QUERIES_WITH_ENTITIES_CSV}")
query_df.head(100)

📄 Queries with locality saved as: q2_queries_with_locations.csv


,query
0,las vegas
1,mexico hotels all inclusive
2,typhoon texas waterpark
3,spa houston
4,dreamhack dallas
5,kalahari
6,dallas cowboys
7,"great wolf lodge dallas, tx"
8,greece
9,resorts texas
